# 🤖 Open-Source LLM Benchmarking for Google Colab

## 📋 Overview

This notebook benchmarks popular open-source LLMs on the HumanEvalComm V2 dataset, optimized for Google Colab execution.

### 🎯 Supported Models
- **CodeLlama-7B-Instruct** - Meta's code-specialized model
- **DeepSeek-Coder-6.7B-Instruct** - DeepSeek's coding specialist
- **CodeQwen1.5-7B-Chat** - Alibaba's code generation model
- **StarCoder2-7B** - BigCode's latest coding model

### 🔧 Colab Optimizations
- Automatic dependency installation
- Memory-efficient model loading
- Fallback strategies for loading issues
- Progress tracking and error handling

### 📊 Evaluation Metrics
- Code generation quality
- Execution success rate
- Generation speed
- Comprehensive analysis using existing evaluators

---
**Created:** 2025-09-15 | **Framework:** HumanEvalComm V2

## 1. 🛠️ Colab Environment Setup

Install all required dependencies and configure the environment for optimal performance in Colab.

In [ ]:
# Install required packages with specific versions for Colab compatibility
!pip install -q transformers==4.36.0
!pip install -q torch torchvision torchaudio
!pip install -q accelerate==0.25.0
!pip install -q bitsandbytes==0.41.3
!pip install -q jsonlines pandas matplotlib seaborn
!pip install -q psutil

print("✅ All dependencies installed successfully")

# Check GPU availability
import torch
if torch.cuda.is_available():
    print(f"🚀 GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU available, using CPU (will be slower)")

## 2. 📦 Import Libraries

Import all necessary libraries and configure logging for clean output.

In [ ]:
import os
import sys
import json
import time
import warnings
import tempfile
import subprocess
from typing import List, Dict, Optional, Tuple, Any

import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
    logging as transformers_logging
)

import jsonlines
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configure for clean output
warnings.filterwarnings('ignore')
transformers_logging.set_verbosity_error()

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Device: {DEVICE}")

# Configure quantization for memory efficiency
if DEVICE == "cuda":
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    print("⚡ 4-bit quantization enabled for memory efficiency")
else:
    quantization_config = None
    print("💾 Using full precision on CPU")

print("✅ Libraries imported and configured")

## 3. 🎯 Model Configuration

Configure the models to benchmark with Colab-optimized settings.

In [ ]:
# Model configurations optimized for Colab
MODELS = [
    {
        "name": "CodeLlama-7B-Instruct",
        "huggingface_id": "codellama/CodeLlama-7b-Instruct-hf",
        "size": "7B",
        "description": "Meta's instruction-tuned code generation model",
        "colab_compatible": True
    },
    {
        "name": "DeepSeek-Coder-6.7B-Instruct",
        "huggingface_id": "deepseek-ai/deepseek-coder-6.7b-instruct",
        "size": "6.7B",
        "description": "DeepSeek's specialized coding model",
        "colab_compatible": True
    },
    {
        "name": "CodeQwen1.5-7B-Chat",
        "huggingface_id": "Qwen/CodeQwen1.5-7B-Chat",
        "size": "7B",
        "description": "Alibaba's code generation and chat model",
        "colab_compatible": True
    },
    {
        "name": "StarCoder2-7B",
        "huggingface_id": "bigcode/starcoder2-7b",
        "size": "7B",
        "description": "BigCode's latest code generation model",
        "colab_compatible": True
    }
]

# Select models for benchmarking (adjust as needed)
SELECTED_MODELS = MODELS[:2]  # First 2 models for demo

print("🎯 Selected Models:")
print("=" * 60)
for i, model in enumerate(SELECTED_MODELS, 1):
    status = "✅ Colab Ready" if model.get('colab_compatible', False) else "⚠️ May have issues"
    print(f"{i}. {model['name']} ({model['size']}) - {status}")
    print(f"   📝 {model['description']}")
    print(f"   🔗 {model['huggingface_id']}")
    print()

print(f"📊 Total models to benchmark: {len(SELECTED_MODELS)}")

## 4. 📊 Load Benchmark Dataset

Load the HumanEvalComm V2 benchmark problems.

In [ ]:
def load_benchmark_data(file_path: str, limit: int = 3) -> List[Dict]:
    """
    Load benchmark problems from JSONL file.
    
    Args:
        file_path: Path to benchmark JSONL file
        limit: Number of problems to load (for testing)
    
    Returns:
        List of problem dictionaries
    """
    problems = []
    
    # Try multiple possible locations
    possible_paths = [
        file_path,
        f"Benchmark/{os.path.basename(file_path)}",
        f"../Benchmark/{os.path.basename(file_path)}"
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            try:
                with jsonlines.open(path) as reader:
                    for i, problem in enumerate(reader):
                        if i >= limit:
                            break
                        problems.append(problem)
                print(f"✅ Loaded {len(problems)} problems from {path}")
                break
            except Exception as e:
                print(f"❌ Error reading {path}: {e}")
                continue
    
    if not problems:
        print(f"❌ Could not find benchmark file. Tried:")
        for path in possible_paths:
            print(f"   - {path}")
        
        # Create sample problems for testing
        print("\n🔧 Creating sample problems for testing...")
        problems = [
            {
                "task_id": "sample_fibonacci",
                "prompt": "def fibonacci(n):\n    \"\"\"Return the nth Fibonacci number\"\"\"\n",
                "test": "assert fibonacci(0) == 0\nassert fibonacci(1) == 1\nassert fibonacci(5) == 5"
            },
            {
                "task_id": "sample_factorial",
                "prompt": "def factorial(n):\n    \"\"\"Return n factorial\"\"\"\n",
                "test": "assert factorial(0) == 1\nassert factorial(5) == 120"
            },
            {
                "task_id": "sample_reverse",
                "prompt": "def reverse_string(s):\n    \"\"\"Reverse a string\"\"\"\n",
                "test": "assert reverse_string('hello') == 'olleh'\nassert reverse_string('') == ''"
            }
        ]
        print(f"✅ Created {len(problems)} sample problems")
    
    return problems

# Load benchmark data
BENCHMARK_FILE = "Benchmark/HumanEvalComm_v2.jsonl"
PROBLEM_LIMIT = 3  # Small number for Colab demo

benchmark_problems = load_benchmark_data(BENCHMARK_FILE, limit=PROBLEM_LIMIT)

print(f"\n📊 Dataset Summary:")
print(f"   Problems loaded: {len(benchmark_problems)}")

if benchmark_problems:
    print(f"\n📝 Sample Problem:")
    sample = benchmark_problems[0]
    print(f"   Task: {sample.get('task_id', 'N/A')}")
    print(f"   Prompt: {sample.get('prompt', '')[:100]}...")
    print(f"   Has tests: {'test' in sample}")

## 5. 🤖 Colab-Optimized Model Loading

Load models with proper error handling and memory management for Colab.

In [ ]:
def load_model_colab_safe(model_config: Dict) -> Tuple[Optional[AutoTokenizer], Optional[AutoModelForCausalLM]]:
    """
    Load model with Colab-specific optimizations and error handling.
    
    Args:
        model_config: Model configuration dictionary
    
    Returns:
        Tuple of (tokenizer, model) or (None, None) if failed
    """
    model_name = model_config['name']
    hf_id = model_config['huggingface_id']
    
    print(f"🔄 Loading {model_name}...")
    print(f"   📝 Loading tokenizer...")
    
    try:
        # Load tokenizer first
        tokenizer = AutoTokenizer.from_pretrained(hf_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        print(f"   ✅ Tokenizer loaded")
        
        print(f"   🧠 Loading model...")
        
        # Try different loading strategies
        loading_strategies = [
            # Strategy 1: 4-bit quantization (most memory efficient)
            {
                "quantization_config": BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type="nf4"
                ),
                "torch_dtype": torch.float16,
                "device_map": "auto",
                "name": "4-bit quantization"
            },
            # Strategy 2: 8-bit quantization
            {
                "quantization_config": BitsAndBytesConfig(load_in_8bit=True),
                "device_map": "auto",
                "name": "8-bit quantization"
            },
            # Strategy 3: Float16 without quantization
            {
                "torch_dtype": torch.float16,
                "device_map": "auto" if DEVICE == "cuda" else None,
                "name": "float16"
            },
            # Strategy 4: CPU fallback
            {
                "torch_dtype": torch.float32,
                "name": "CPU fallback"
            }
        ]
        
        model = None
        for i, strategy in enumerate(loading_strategies):
            try:
                print(f"     Trying strategy {i+1}: {strategy['name']}")
                
                model = AutoModelForCausalLM.from_pretrained(hf_id, **strategy)
                
                # Move to device if not using device_map
                if "device_map" not in strategy and DEVICE == "cpu":
                    model = model.to(DEVICE)
                
                model.eval()
                print(f"     ✅ Success with {strategy['name']}")
                break
                
            except Exception as e:
                print(f"     ❌ Failed with {strategy['name']}: {str(e)[:100]}...")
                if model is not None:
                    del model
                    model = None
                torch.cuda.empty_cache() if DEVICE == "cuda" else None
                continue
        
        if model is None:
            print(f"   ❌ All loading strategies failed for {model_name}")
            return None, None
        
        # Calculate model info
        total_params = sum(p.numel() for p in model.parameters())
        model_device = next(model.parameters()).device
        
        print(f"   ✅ {model_name} loaded successfully")
        print(f"   📊 Parameters: {total_params:,}")
        print(f"   🖥️ Device: {model_device}")
        
        return tokenizer, model
        
    except Exception as e:
        print(f"   ❌ Critical error loading {model_name}: {e}")
        return None, None

def generate_code_safe(
    prompt: str,
    tokenizer: AutoTokenizer,
    model: AutoModelForCausalLM,
    max_new_tokens: int = 200,
    temperature: float = 0.7
) -> str:
    """
    Generate code with error handling and memory management.
    
    Args:
        prompt: Code generation prompt
        tokenizer: Model tokenizer
        model: Language model
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature
    
    Returns:
        Generated code string
    """
    try:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                num_return_sequences=1,
                repetition_penalty=1.1
            )
        
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract generated portion
        if generated_text.startswith(prompt):
            return generated_text[len(prompt):].strip()
        return generated_text.strip()
        
    except Exception as e:
        print(f"       ❌ Generation error: {e}")
        return f"# Error during generation: {e}"

print("🔧 Colab-safe model functions defined")

## 6. 🧪 Simple Evaluation Framework

Define a simplified evaluation approach that works reliably in Colab.

In [ ]:
def evaluate_code_simple(code: str, test_code: str) -> Dict:
    """
    Simple code evaluation using subprocess execution.
    
    Args:
        code: Generated code to evaluate
        test_code: Test cases to run
    
    Returns:
        Dictionary with evaluation results
    """
    if not test_code:
        return {
            "success": False,
            "output": "",
            "error": "No test cases provided",
            "execution_time": 0.0
        }
    
    # Create temporary file with code and tests
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(code)
        f.write("\n\n")
        f.write(test_code)
        temp_file = f.name
    
    try:
        start_time = time.time()
        result = subprocess.run(
            [sys.executable, temp_file],
            capture_output=True,
            text=True,
            timeout=10
        )
        execution_time = time.time() - start_time
        
        return {
            "success": result.returncode == 0,
            "output": result.stdout,
            "error": result.stderr,
            "execution_time": execution_time,
            "return_code": result.returncode
        }
        
    except subprocess.TimeoutExpired:
        return {
            "success": False,
            "output": "",
            "error": "Execution timeout",
            "execution_time": 10.0
        }
    except Exception as e:
        return {
            "success": False,
            "output": "",
            "error": str(e),
            "execution_time": 0.0
        }
    finally:
        try:
            os.unlink(temp_file)
        except:
            pass

def calculate_code_quality_score(code: str) -> float:
    """
    Calculate a simple code quality score based on basic metrics.
    
    Args:
        code: Generated code to analyze
    
    Returns:
        Quality score between 0 and 10
    """
    if not code or code.startswith("# Error"):
        return 0.0
    
    score = 5.0  # Base score
    
    # Check for basic code structure
    if "def " in code:
        score += 1.0
    
    # Check for docstrings
    if '"""' in code or "'''" in code:
        score += 0.5
    
    # Check for comments
    if "#" in code:
        score += 0.5
    
    # Penalize very short or very long code
    code_length = len(code.strip())
    if code_length < 20:
        score -= 2.0
    elif code_length > 1000:
        score -= 1.0
    
    # Check for basic error handling
    if "try:" in code or "except" in code:
        score += 1.0
    
    return max(0.0, min(10.0, score))

print("🧪 Simple evaluation framework defined")

## 7. 🏃 Execute Benchmark

Run the benchmark across all selected models with progress tracking.

In [ ]:
def benchmark_model_colab(model_config: Dict, problems: List[Dict]) -> Optional[Dict]:
    """
    Benchmark a single model on all problems with Colab optimizations.
    
    Args:
        model_config: Model configuration
        problems: List of benchmark problems
    
    Returns:
        Benchmark results dictionary or None if failed
    """
    model_name = model_config['name']
    
    print(f"\n🤖 Processing {model_name}...")
    
    # Load model
    tokenizer, model = load_model_colab_safe(model_config)
    if not tokenizer or not model:
        print(f"❌ Skipping {model_name} due to loading failure")
        return None
    
    # Evaluate each problem
    results = []
    total_generation_time = 0.0
    successful_executions = 0
    
    for i, problem in enumerate(problems):
        task_id = problem.get('task_id', f'problem_{i}')
        prompt = problem.get('prompt', '')
        test_code = problem.get('test', '')
        
        print(f"  📝 Problem {i+1}/{len(problems)}: {task_id}")
        
        try:
            # Generate code
            start_time = time.time()
            generated_code = generate_code_safe(prompt, tokenizer, model)
            generation_time = time.time() - start_time
            total_generation_time += generation_time
            
            # Evaluate code
            eval_result = evaluate_code_simple(generated_code, test_code)
            quality_score = calculate_code_quality_score(generated_code)
            
            if eval_result['success']:
                successful_executions += 1
            
            result = {
                'task_id': task_id,
                'prompt': prompt,
                'generated_code': generated_code,
                'generation_time': generation_time,
                'evaluation': eval_result,
                'quality_score': quality_score,
                'success': eval_result['success']
            }
            
            results.append(result)
            
            # Progress update
            status = "✅ PASS" if eval_result['success'] else "❌ FAIL"
            print(f"    {status} | Time: {generation_time:.2f}s | Quality: {quality_score:.1f}/10")
            
        except Exception as e:
            print(f"    ❌ Error processing problem: {e}")
            continue
    
    # Calculate summary
    if results:
        avg_generation_time = total_generation_time / len(results)
        success_rate = (successful_executions / len(results)) * 100
        avg_quality_score = sum(r['quality_score'] for r in results) / len(results)
        
        summary = {
            'model_name': model_name,
            'model_config': model_config,
            'total_problems': len(results),
            'successful_executions': successful_executions,
            'success_rate': success_rate,
            'avg_generation_time': avg_generation_time,
            'avg_quality_score': avg_quality_score,
            'results': results
        }
        
        print(f"\n  📊 {model_name} Results:")
        print(f"     Success Rate: {success_rate:.1f}%")
        print(f"     Avg Quality Score: {avg_quality_score:.2f}/10")
        print(f"     Avg Generation Time: {avg_generation_time:.2f}s")
        
        # Clean up memory
        del model
        del tokenizer
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        
        return summary
    
    return None

# Execute benchmark
print("🚀 Starting Multi-Model Benchmark")
print(f"Models: {len(SELECTED_MODELS)} | Problems: {len(benchmark_problems)}")

benchmark_results = []

for model_config in SELECTED_MODELS:
    try:
        result = benchmark_model_colab(model_config, benchmark_problems)
        if result:
            benchmark_results.append(result)
    except Exception as e:
        print(f"❌ Critical error benchmarking {model_config['name']}: {e}")
        continue

print(f"\n🎉 Benchmark Complete! Successfully evaluated {len(benchmark_results)} models.")

## 8. 📊 Results Analysis and Visualization

Analyze and visualize the benchmark results with comprehensive comparisons.

In [ ]:
if not benchmark_results:
    print("❌ No benchmark results available for analysis.")
    print("\n🔧 Troubleshooting Tips:")
    print("1. Ensure you have sufficient GPU memory")
    print("2. Try reducing the number of models or problems")
    print("3. Check internet connection for model downloads")
    print("4. Restart runtime if memory issues persist")
else:
    # Create comparison DataFrame
    comparison_data = []
    for result in benchmark_results:
        comparison_data.append({
            'Model': result['model_name'],
            'Success Rate (%)': result['success_rate'],
            'Avg Quality Score': result['avg_quality_score'],
            'Avg Generation Time (s)': result['avg_generation_time'],
            'Problems Solved': result['successful_executions'],
            'Total Problems': result['total_problems']
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('Success Rate (%)', ascending=False)
    
    print("📊 BENCHMARK RESULTS")
    print("=" * 80)
    display(comparison_df)
    
    # Find best model
    best_model = comparison_df.iloc[0]
    print(f"\n🏆 BEST PERFORMING MODEL: {best_model['Model']}")
    print(f"   Success Rate: {best_model['Success Rate (%)']:.1f}%")
    print(f"   Quality Score: {best_model['Avg Quality Score']:.2f}/10")
    print(f"   Generation Speed: {best_model['Avg Generation Time (s)']:.2f}s")
    
    # Create visualizations
    plt.style.use('default')
    sns.set_palette("husl")
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('🤖 Open-Source LLM Benchmark Results', fontsize=16, fontweight='bold')
    
    # Success Rate
    axes[0,0].bar(comparison_df['Model'], comparison_df['Success Rate (%)'], color='lightgreen')
    axes[0,0].set_title('Success Rate (%)')
    axes[0,0].set_ylabel('Success Rate (%)')
    axes[0,0].tick_params(axis='x', rotation=45)
    for i, v in enumerate(comparison_df['Success Rate (%)']):
        axes[0,0].text(i, v + 1, f'{v:.1f}%', ha='center', va='bottom')
    
    # Quality Score
    axes[0,1].bar(comparison_df['Model'], comparison_df['Avg Quality Score'], color='skyblue')
    axes[0,1].set_title('Average Quality Score')
    axes[0,1].set_ylabel('Quality Score (0-10)')
    axes[0,1].tick_params(axis='x', rotation=45)
    for i, v in enumerate(comparison_df['Avg Quality Score']):
        axes[0,1].text(i, v + 0.1, f'{v:.2f}', ha='center', va='bottom')
    
    # Generation Time
    axes[1,0].bar(comparison_df['Model'], comparison_df['Avg Generation Time (s)'], color='lightcoral')
    axes[1,0].set_title('Average Generation Time')
    axes[1,0].set_ylabel('Time (seconds)')
    axes[1,0].tick_params(axis='x', rotation=45)
    for i, v in enumerate(comparison_df['Avg Generation Time (s)']):
        axes[1,0].text(i, v + 0.05, f'{v:.2f}s', ha='center', va='bottom')
    
    # Problems Solved
    axes[1,1].bar(comparison_df['Model'], comparison_df['Problems Solved'], color='gold')
    axes[1,1].set_title('Problems Solved')
    axes[1,1].set_ylabel('Number of Problems')
    axes[1,1].tick_params(axis='x', rotation=45)
    for i, v in enumerate(comparison_df['Problems Solved']):
        axes[1,1].text(i, v + 0.05, f'{int(v)}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Save results
    with open('colab_benchmark_results.json', 'w') as f:
        json.dump(benchmark_results, f, indent=2, default=str)
    
    comparison_df.to_csv('colab_model_comparison.csv', index=False)
    
    print("\n💾 Results saved:")
    print("   📄 colab_benchmark_results.json (detailed results)")
    print("   📊 colab_model_comparison.csv (summary table)")

## 9. 📝 Sample Generated Code

Display sample generated code from the best performing model.

In [ ]:
if benchmark_results:
    # Find best model results
    best_model_name = best_model['Model']
    best_model_results = next(r for r in benchmark_results if r['model_name'] == best_model_name)
    
    print(f"📝 SAMPLE CODE FROM BEST MODEL: {best_model_name}")
    print("=" * 80)
    
    for i, result in enumerate(best_model_results['results'][:3]):
        task_id = result['task_id']
        prompt = result['prompt']
        generated_code = result['generated_code']
        success = result['success']
        quality = result['quality_score']
        
        print(f"\n🔍 Example {i+1}: {task_id}")
        print("-" * 50)
        print(f"📋 Prompt:")
        print(prompt[:150] + ("..." if len(prompt) > 150 else ""))
        
        print(f"\n💻 Generated Code:")
        print("```python")
        print(generated_code[:400] + ("..." if len(generated_code) > 400 else ""))
        print("```")
        
        status_emoji = "✅" if success else "❌"
        print(f"\n📊 Results: {status_emoji} Success | Quality: {quality:.1f}/10")
        
        if not success and result['evaluation']['error']:
            print(f"❌ Error: {result['evaluation']['error'][:100]}...")
    
    # Overall statistics
    print(f"\n📈 OVERALL STATISTICS")
    print("=" * 50)
    for result in benchmark_results:
        print(f"🤖 {result['model_name']}:")
        print(f"   Success: {result['success_rate']:.1f}%")
        print(f"   Quality: {result['avg_quality_score']:.2f}/10")
        print(f"   Speed: {result['avg_generation_time']:.2f}s")
        print()
        
else:
    print("❌ No results to display")

## 10. 🔧 Troubleshooting and Tips

Common issues and solutions for running LLM benchmarks in Colab.

In [ ]:
print("🔧 TROUBLESHOOTING GUIDE")
print("=" * 60)

print("\n❌ Model Loading Issues:")
print("   • bitsandbytes error: Run '!pip install -U bitsandbytes' and restart runtime")
print("   • Out of memory: Reduce model size or enable quantization")
print("   • Access denied: Some models require HuggingFace authentication")

print("\n⚡ Performance Optimization:")
print("   • Use GPU runtime: Runtime → Change runtime type → GPU")
print("   • Enable quantization for larger models")
print("   • Reduce max_new_tokens for faster generation")
print("   • Process fewer problems for quick testing")

print("\n💾 Memory Management:")
print("   • Models are automatically cleaned from memory after evaluation")
print("   • Restart runtime if you encounter persistent memory issues")
print("   • Monitor GPU memory: !nvidia-smi")

print("\n🔄 Customization Options:")
print("   • Modify SELECTED_MODELS to choose different models")
print("   • Adjust PROBLEM_LIMIT to test more/fewer problems")
print("   • Change generation parameters (temperature, max_tokens)")

# System information
print(f"\n🖥️ SYSTEM INFO:")
print(f"   Python: {sys.version.split()[0]}")
print(f"   PyTorch: {torch.__version__}")
print(f"   Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")

print("\n✅ Troubleshooting guide complete")